### Merging observation data for classic evaluation

In [6]:
# Importing libraries
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import glob
import os

In [7]:
# Paths
input_folder = '/home/rosquete/Documents/FRESH-CARE/data/fusion_evaluation/classic_evaluation/in_situ_data/data_ready/'
output_drifters_file = '/home/rosquete/Documents/FRESH-CARE/data/fusion_evaluation/classic_evaluation/in_situ_data/drifters_merged.parquet'
arctic_regions_file = '/home/rosquete/Documents/FRESH-CARE/regions/delimitation/arctic_regions.json'

In [8]:
# Merging all drifter data into one df

def merge_drifter_data(folder_path):

    all_files = []
    files = glob.glob(os.path.join(folder_path, "*.csv"))

    for f in files:
        try:
            with open(f, 'r') as file:
                lines = file.readlines()

            # find the header line dynamically (robust to variable metadata length)
            header_idx = next(
                (i for i, l in enumerate(lines) if l.lstrip().lower().startswith('time,')),
                None
            )
            if header_idx is None:
                print(f"Skipping {f}: no header found")
                continue

            # extract drifter ID 
            id_drifter = lines[0].split(',')[1].strip()
            dftemp = pd.read_csv(f, skiprows=header_idx)
            dftemp['id'] = id_drifter

            # time column to datetime format
            dftemp['time'] = pd.to_datetime(dftemp['time'])

            # getting columns of interest
            columnas_ok = ['id', 'time', 'latitude', 'lon360', 've', 'vn', 'speed', 'direction']
            dftemp = dftemp[[c for c in columnas_ok if c in dftemp.columns]]

            all_files.append(dftemp)

        except Exception as e:
            print(f"Error reading {f}: {e}")

    # concatenate all dataFrames
    df_merged = pd.concat(all_files, ignore_index=True)
    return df_merged

df_obs = merge_drifter_data(input_folder)
print (df_obs.head())
print(f"All records loaded: {len(df_obs)}")


       id                      time  latitude     lon360        ve        vn  \
0  104126 2011-11-15 00:00:00+00:00  54.14250  198.97375 -0.083290 -0.065552   
1  104126 2011-11-16 00:00:00+00:00  54.08925  198.79750 -0.166035 -0.067110   
2  104126 2011-11-17 00:00:00+00:00  54.04275  198.56950 -0.190586 -0.043945   
3  104126 2011-11-18 00:00:00+00:00  54.01025  198.29325 -0.240829 -0.064287   
4  104126 2011-11-19 00:00:00+00:00  53.98300  197.94225 -0.253266  0.017257   

      speed   direction  
0  0.105992  231.795925  
1  0.179085  247.991706  
2  0.195587  257.015818  
3  0.249261  255.053931  
4  0.253853  273.897887  
All records loaded: 110283


In [9]:
# Assigning Arctic regions to drifter data

# change longitudes to -180 to 180 format for spatial join
df_obs['lon_180'] = ((df_obs['lon360'] + 180) % 360) - 180

# create a GeoDataFrame from the drifters
geometry = [Point(xy) for xy in zip(df_obs['lon_180'], df_obs['latitude'])]
gdf_drifters = gpd.GeoDataFrame(df_obs, geometry=geometry)

# set the coordinate reference system to WGS84 (standard lat/lon)
gdf_drifters.set_crs(epsg=4326, inplace=True)

# loading the Arctic regions
gdf_regions = gpd.read_file(arctic_regions_file)

# spatial join to assign region IDs to drifter points
df_labeled = gpd.sjoin(gdf_drifters, gdf_regions[['id', 'geometry']], how='left', predicate='within')

# handle drifters outside the sectors
df_labeled['region'] = df_labeled['id_right'].fillna('Out')


In [10]:
# Saving final merged and labeled drifter data

df_final = df_labeled[[
    'id_left', 'time', 'latitude', 'lon_180','lon360', 've', 'vn', 'region'
]].rename(columns={
    'id_left': 'id_drifter',
    'region': 'id_sector',
    'lon_180': 'longitude'
})

# Save the final labeled dataset
df_final.to_parquet(output_drifters_file)

print("Labeling complete!")
print(df_final['id_drifter'].value_counts())

Labeling complete!
id_drifter
300234066218830    945
300234065719120    913
300234066515010    756
300234066416870    756
300234062885800    720
                  ... 
81922                2
300234062417590      2
132594               2
300234068346190      2
81919                2
Name: count, Length: 742, dtype: int64
